# A guide Portfolio Optimization Environment

This notebook aims to provide an example of using PortfolioOptimizationEnv (or POE) to train a reinforcement learning model that learns to solve the portfolio optimization problem.

In this document, we will reproduce a famous architecture called EIIE (ensemble of identical independent evaluators), introduced in the following paper:

- Zhengyao Jiang, Dixing Xu, & Jinjun Liang. (2017). A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem. https://doi.org/10.48550/arXiv.1706.10059.

It's advisable to read it to understand the algorithm implemented in this notebook.

### Note
If you're using this environment, consider citing the following paper (in adittion to FinRL references):

- Caio Costa, & Anna Costa (2023). POE: A General Portfolio Optimization Environment for FinRL. In *Anais do II Brazilian Workshop on Artificial Intelligence in Finance* (pp. 132–143). SBC. https://doi.org/10.5753/bwaif.2023.231144.

```
@inproceedings{bwaif,
 author = {Caio Costa and Anna Costa},
 title = {POE: A General Portfolio Optimization Environment for FinRL},
 booktitle = {Anais do II Brazilian Workshop on Artificial Intelligence in Finance},
 location = {João Pessoa/PB},
 year = {2023},
 keywords = {},
 issn = {0000-0000},
 pages = {132--143},
 publisher = {SBC},
 address = {Porto Alegre, RS, Brasil},
 doi = {10.5753/bwaif.2023.231144},
 url = {https://sol.sbc.org.br/index.php/bwaif/article/view/24959}
}

```

## Installation and imports

To run this notebook in google colab, uncomment the cells below.

In [ ]:
## install finrl library
# !sudo apt install swig
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.2.0-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 25 not upgraded.


In [ ]:
## We also need to install quantstats, because the environment uses it to plot graphs
# !pip install quantstats

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/cc/74/b9cf9a2cf911d48c120635b71281a3a28323e97813e7a4d459b6acfd6447/QuantStats-0.0.64-py2.py3-none-any.whl (45 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/40/44/4a5f08c96eb108af5cb50b41f76142f0afa346dfa99d5296fe7202a11854/tabulate-0.9.0-py3-none-any.whl (35 kB)


In [4]:
## Hide matplotlib warnings
# import warnings
# warnings.filterwarnings('ignore')

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

#### Import the necessary code libraries

In [5]:
import pandas as pd
import yfinance as yf
from datetime import datetime
class YahooRealtimeDownloader:
    """
    Provides methods for retrieving near-real-time (1-minute) stock data
    from Yahoo Finance API, returning only the last available bar or
    "one minute behind" the latest bar.
    """

    def __init__(self, ticker_list: list):
        """
        Parameters
        ----------
        ticker_list: list
            a list of stock tickers
        """
        self.ticker_list = ticker_list

    def fetch_data(self, proxy=None, pick_second_to_last=True) -> pd.DataFrame:
        """
        Fetches near-real-time 1-minute data from Yahoo API for the current day.
        """
        import datetime
        data_df = pd.DataFrame()
        num_failures = 0

        # 按照 1 分钟周期下载当日数据
        for tic in self.ticker_list:
            temp_df = yf.download(
                tickers=tic,
                period='1d',
                interval='1m',
                proxy=proxy,
                progress=False
            )

            temp_df["tic"] = tic

            if len(temp_df) > 0:
                # 如果您想获取倒数第二条数据
                if pick_second_to_last and len(temp_df) > 1:
                    temp_df = temp_df.iloc[[-2]]  # 取倒数第二行
                else:
                    temp_df = temp_df.iloc[[-1]]

                data_df = pd.concat([data_df, temp_df], axis=0)
            else:
                num_failures += 1

        if num_failures == len(self.ticker_list):
            raise ValueError("No data is fetched. Possibly all tickers returned empty for today.")

        # reset the index
        data_df = data_df.reset_index()

        # rename columns
        try:
            data_df.columns = [
                "date",
                "open",
                "high",
                "low",
                "close",
                "adjcp",
                "volume",
                "tic",
            ]
            data_df["close"] = data_df["adjcp"]
            data_df = data_df.drop(labels="adjcp", axis=1)
        except ValueError:
            print("Columns might not match the expected format; please check yfinance returned columns.")

        # 将日期列转换为 datetime
        data_df["date"] = pd.to_datetime(data_df["date"])
        
        # 获取本地当前日期，例如 2023-10-10
        today = datetime.date.today()

        # 如果行里的 date 不是当天，就改为当天；去掉时分秒，保留 YYYY-MM-DD
        def fix_date(dt):
            if dt.date() != today:
                return today.strftime('%Y-%m-%d')
            else:
                return dt.strftime('%Y-%m-%d')

        data_df["date"] = data_df["date"].apply(fix_date)

        # 再创建 day 列，这里仅保留日期而无时分秒，所以先转回 datetime
        data_df["date"] = pd.to_datetime(data_df["date"])
        data_df["day"] = data_df["date"].dt.dayofweek
        data_df["date"] = data_df["date"].dt.strftime("%Y-%m-%d")

        data_df = data_df.dropna().reset_index(drop=True)

        data_df = data_df.sort_values(by=["date", "tic"]).reset_index(drop=True)

        print("Shape of realtime DataFrame: ", data_df.shape)
        print(data_df)
        return data_df

In [8]:
from IPython import get_ipython

ip = get_ipython()
if not hasattr(ip, 'magic'):
    ip.magic = lambda x: None  # 或者提供更加完整的实现
import torch

import numpy as np

from sklearn.preprocessing import MaxAbsScaler

import sys
import os

# 获取当前文件的绝对路径，并向上追溯两级到项目根目录
project_root = os.path.dirname(os.path.abspath(os.getcwd()))
print(project_root)
sys.path.append(project_root)

from finrl.meta.preprocessor.tusharedownloader import TushareDownloader
from finrl.meta.preprocessor.preprocessors import GroupByScaler
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl.agents.portfolio_optimization.models import DRLAgent
from finrl.agents.portfolio_optimization.architectures import EIIE

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

/home/stock/projects/finstock/FinRL


## Fetch data

In his paper, *Jiang et al* creates a portfolio composed by the top-11 cryptocurrencies based on 30-days volume. Since it's not specified when this classification was done, it's difficult to reproduce, so we will use a similar approach in the Brazillian stock market:

- We select top-10 stocks from Brazillian stock market;
- For simplicity, we disconsider stocks that have missing data for the days in period 2011-01-01 to 2019-12-31 (9 years);

In [2]:
market = "ch"
if market.lower() == "us":
    TOP_BRL = [
        'BILI', 'DUO', 'NIO', 'JD', 'YINN', 'YANG', 'FUTU','XHG','BABA','PDD','RGTI'
    ]
elif market.lower() == "hk":
    TOP_BRL = [
         '1812.hk', '3900.hk', '2777.hk', '1810.hk', '2878.HK','0029.hk'
    ]
elif market.lower() == "ch":
    TOP_BRL = [
         '603063.ss', '603319.ss', '000657.sz','002640.sz','002664.sz','300182.sz','688200.SS','002850.SZ'
    ]

    # TOP_BRL = [
    # '000063.SZ',
    # '002068.SZ',
    # '002138.SZ',
    # '002779.SZ',
    # '002850.SZ',
    # '300408.SZ',
    # '300442.SZ',
    # '300657.SZ',
    # '300673.SZ',
    # '300909.SZ',
    # '300913.SZ',
    # '601111.SS',
    # '601689.SS',
    # '603063.SS',
    # '603236.SS',
    # '603305.SS',
    # '603556.SS',
    # '603667.SS',
    # '688088.SS',
    # '688160.SS',
    # '688200.SS']
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

print("当前选择的市场:", market)
print("股票列表:", TOP_BRL)
# '9988.hk',,'1797.hk''1918.hk', '3319.hk',

当前选择的市场: ch
股票列表: ['603063.ss', '603319.ss', '000657.sz', '002640.sz', '002664.sz', '300182.sz', '688200.SS', '002850.SZ']


In [ ]:
# 原有数据获取方式（保持不变）
print(len(TOP_BRL))

portfolio_raw_df = TushareDownloader(start_date = '2022-01-01',
                                end_date = '2025-03-26',
                                ticker_list = TOP_BRL).fetch_data()

# 展示如何在未来使用 PoE 数据管道
print("\n" + "="*60)
print("🚀 如何使用 PoE 数据管道 (未来升级)")
print("="*60)
print("""
# 1. 使用 PoE 数据管道和过滤器的示例代码:
from poe.core.config import FinRLConfig
from poe.core.protfolioexperiment import PortfolioOptimizationExperiment

# 2. 配置 PoE (包含节假日过滤)
poe_config = FinRLConfig(
    top_brl=TOP_BRL,
    features=["close", "high", "low", "volume"],
    time_window=4,
    user_id=1,
    group_id=1,
    date_range={
        "train_start": "2022-01-01",
        "train_end": "2024-03-01", 
        "test_start": "2024-03-01",
        "test_end": "2025-03-26"
    },
    # 节假日过滤器 (插件化)
    date_filters=[
        {
            "name": "chinese_holidays",
            "config": {
                "days_before": 15,    # 节前15天
                "days_after": 10,     # 节后10天
                "holidays": ['春节', '五一', '国庆']
            }
        }
    ]
)

# 3. 获取处理好的数据
exp = PortfolioOptimizationExperiment(poe_config)
env_train, env_test, df_train, df_test = exp._setup_training_environment()
portfolio_raw_df = pd.concat([df_train, df_test], ignore_index=True)
""")
print("="*60)

portfolio_raw_df

In [11]:
# realtime_df = YahooRealtimeDownloader(ticker_list = TOP_BRL).fetch_data()
# 假设这两个 DataFrame 的列名相同 
# portfolio_raw_df = pd.concat([portfolio_raw_df, realtime_df], ignore_index=True)

# 对合并后的数据，根据日期和股票标的排序，并重新索引
portfolio_raw_df = portfolio_raw_df.sort_values(["date", "tic"]).reset_index(drop=True)

# 检查合并后的数据
print(portfolio_raw_df.head())
print(portfolio_raw_df.tail())

         date    open    high     low   close      volume        tic  day
0  2022-01-04   16.08   16.19   15.20   15.22   264158.22  000657.sz    1
1  2022-01-04    3.08    3.29    3.08    3.29   535143.14  002640.sz    1
2  2022-01-04   20.30   20.78   20.01   20.26    41189.89  002664.sz    1
3  2022-01-04  164.00  164.00  151.51  153.20    28140.37  002850.SZ    1
4  2022-01-04    6.60    6.77    6.36    6.54  3033080.78  300182.sz    1
            date    open    high     low   close      volume        tic  day
4340  2024-12-31    4.04    4.10    3.80    3.84  1457282.32  002640.sz    1
4341  2024-12-31   14.22   14.59   13.99   14.06    96976.00  002664.sz    1
4342  2024-12-31   98.91   99.35   95.92   97.68    28392.89  002850.SZ    1
4343  2024-12-31    6.27    6.32    5.85    5.87  1159665.50  300182.sz    1
4344  2024-12-31  105.13  107.78  104.00  104.50    31618.92  688200.SH    1


In [12]:
from finrl.config import INDICATORS
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,
    use_vix=False,
    use_turbulence=False,
    user_defined_feature=False
)
processed = fe.preprocess_data(portfolio_raw_df)
logging.info("数据预处理完成。")

基准日期范围从 2022-01-04 到 2024-12-31
merged_closes 的列名: Index(['000657.sz', '002640.sz', '002664.sz', '002850.SZ', '300182.sz',
       '688200.SH'],
      dtype='object', name='tic')
merged_closes 的索引名: None
reset_index 后的列名: Index(['date', '000657.sz', '002640.sz', '002664.sz', '002850.SZ', '300182.sz',
       '688200.SH'],
      dtype='object', name='tic')
最终股票代码 000657.sz 的日期一致，日期数量: 726
最终股票代码 002640.sz 的日期一致，日期数量: 726
最终股票代码 002664.sz 的日期一致，日期数量: 726
最终股票代码 002850.SZ 的日期一致，日期数量: 726
最终股票代码 300182.sz 的日期一致，日期数量: 726
最终股票代码 688200.SH 的日期一致，日期数量: 726
Successfully added technical indicators


In [ ]:
# from feature.ch_feature_engineer import ChFeatureEngineer
# ch_fe = ChFeatureEngineer()
# portfolio_processed = ch_fe.preprocess_data(processed)
# logging.info("自定义特征工程完成。")

In [ ]:
# PoE 数据管道已经应用了节假日过滤器，直接使用处理好的数据
print("✅ PoE 数据管道已自动应用节假日过滤器")
print("🎊 节假日过滤配置: 春节、五一、国庆前15天+后10天")

# 数据已经通过 PoE 管道处理，包含节假日过滤
filtered_df = portfolio_raw_df.copy()

print(f"📊 过滤后数据: {len(filtered_df)} 行")
print(f"📅 日期范围: {filtered_df['date'].min()} 至 {filtered_df['date'].max()}")

# 显示股票数据统计
print("\n📊 各股票数据统计:")
print(filtered_df.groupby("tic").count())

In [14]:
filtered_df.groupby("tic").count()

,date,open,high,low,close,volume,day
tic,,,,,,,
000657.sz,189,189,189,189,189,189,189
002640.sz,189,189,189,189,189,189,189
002664.sz,189,189,189,189,189,189,189
002850.SZ,189,189,189,189,189,189,189
300182.sz,189,189,189,189,189,189,189
688200.SH,189,189,189,189,189,189,189


### Normalize Data

We normalize the data dividing the time series of each stock by its maximum value, so that the dataframe contains values between 0 and 1.

In [ ]:
portfolio_norm_df = GroupByScaler(by="tic", scaler=MaxAbsScaler).fit_transform(filtered_df)
portfolio_norm_df

/home/stock/projects/finstock/FinRL/finrl/meta/preprocessor/preprocessors.py:101: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75
 1.   0.   0.25 0.5  0.75 1.   0.   0.75 1.   0.   0.25 0.5  0.75 1.
 0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75
 0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75
 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5
 0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.75 1.
 0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75
 1.   0.   0.25 0.5  0.75 1.   0.   0.   0.25 0.5  0.75 1.   0.   0.25
 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.
 0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.
 0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75 1.   0.   0.25 0.5  0.75
 1.   0.   0.25 0

,date,open,high,low,close,volume,tic,day
0,2022-02-07,0.770122,0.766212,0.755855,0.751728,0.220435,000657.sz,0.00
1,2022-02-07,0.647170,0.634234,0.638258,0.629630,0.129204,002640.sz,0.00
2,2022-02-07,0.907989,0.852209,0.913408,0.856848,0.040349,002664.sz,0.00
3,2022-02-07,0.982061,0.992617,0.973118,0.948001,0.221950,002850.SZ,0.00
4,2022-02-07,0.719498,0.708705,0.795973,0.771144,0.408817,300182.sz,0.00
...,...,...,...,...,...,...,...,...
1129,2024-11-05,0.569811,0.587387,0.562500,0.583333,0.478685,002640.sz,0.25
1130,2024-11-05,0.906336,0.835449,0.864246,0.850129,1.000000,002664.sz,0.25
1131,2024-11-05,0.652353,0.682405,0.661712,0.685001,0.658451,002850.SZ,0.25
1132,2024-11-05,0.632839,0.645089,0.743624,0.713930,0.263015,300182.sz,0.25


In [16]:
df_portfolio = portfolio_norm_df[["date", "tic", "close", "high", "low",'volume']]

df_portfolio_train = df_portfolio[(df_portfolio["date"] >= "2019-01-01") & (df_portfolio["date"] <= "2025-01-29")]

df_portfolio_2025 = df_portfolio[(df_portfolio["date"] >= "2021-09-01") & (df_portfolio["date"] <= "2025-04-15")]

unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)
print(unique_dates)

0      2022-02-07
1      2022-02-08
2      2022-02-09
3      2022-02-10
4      2022-02-11
          ...    
184    2024-10-30
185    2024-10-31
186    2024-11-01
187    2024-11-04
188    2024-11-05
Name: date, Length: 189, dtype: object


### Instantiate Environment

Using the `PortfolioOptimizationEnv`, it's easy to instantiate a portfolio optimization environment for reinforcement learning agents. In the example below, we use the dataframe created before to start an environment.

In [17]:
features=["close", "high", "low",'volume']
environment = PortfolioOptimizationEnv(
        df_portfolio_train,
        initial_amount=100000,
        comission_fee_pct=0.0025,
        time_window=4,
        features=features,
        normalize_df=None
    )

### Instantiate Model

Now, we can instantiate the model using FinRL API. In this example, we are going to use the EIIE architecture introduced by Jiang et. al.

:exclamation: **Note:** Remember to set the architecture's `time_window` parameter with the same value of the environment's `time_window`.

In [21]:
# set PolicyGradient parameters
model_kwargs = {
    "lr": 0.01,
    "policy": EIIE,
}

# here, we can set EIIE's parameters
policy_kwargs = {
    "k_size": 3,
    "time_window": 4,
    "initial_features":len(features)
}

model = DRLAgent(environment).get_model("pg", device, model_kwargs, policy_kwargs)

model_name = "AFTER"
file_path = f"policy_EIIE_US_{model_name}.pt"
import os
import torch

# if market.lower() == "us":
#     if os.path.isfile(file_path):
#         model.train_policy.load_state_dict(torch.load(file_path))
#         print(f"成功加载模型参数：{file_path}")
#     else:
#         print(f"未找到模型文件: {file_path}，请确认路径或先进行保存。")
# elif market.lower() == "hk":
#     if os.path.isfile(file_path):
#         model.train_policy.load_state_dict(torch.load("policy_EIIE_HK.pt"))
#         print("成功加载模型参数：policy_EIIE_HK.pt")
#     else:
#         print(f"未找到模型文件: {file_path}，请确认路径或先进行保存。")
# elif market.lower() == "ch":
#     # model.train_policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
#     # print("成功加载模型参数：policy_EIIE_CH3.pt")
# else:
#     raise ValueError("market 变量必须为 'us' 或 'hk'")

### Train Model

In [25]:
DRLAgent.train_model(model, episodes=600)

if market.lower() == "us":  
    torch.save(model.train_policy.state_dict(), "policy_EIIE_US_{}.pt".format(model_name))

elif market.lower() == "hk":
    torch.save(model.train_policy.state_dict(), "policy_EIIE_HK.pt")
elif market.lower() == "ch":
    torch.save(model.train_policy.state_dict(), "policy_EIIE_CH3.pt")
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

  0%|          | 0/600 [00:00<?, ?it/s]

Initial portfolio value:100000
Final portfolio value: 392846.53125
Final accumulative portfolio value: 3.9284653663635254
Maximum DrawDown: -0.30667698727407655
Sharpe ratio: 2.7792933350894637


  0%|          | 1/600 [00:01<11:45,  1.18s/it]

Initial portfolio value:100000
Final portfolio value: 397294.8125
Final accumulative portfolio value: 3.9729480743408203
Maximum DrawDown: -0.3028695082396581
Sharpe ratio: 2.805202940851459


  0%|          | 2/600 [00:03<17:41,  1.77s/it]

...

100%|█████████▉| 598/600 [12:14<00:02,  1.20s/it]

Initial portfolio value:100000
Final portfolio value: 895997.9375
Final accumulative portfolio value: 8.959979057312012
Maximum DrawDown: -0.14730899293596356
Sharpe ratio: 3.6690299684336987


100%|█████████▉| 599/600 [12:15<00:01,  1.19s/it]

Initial portfolio value:100000
Final portfolio value: 896195.4375
Final accumulative portfolio value: 8.961954116821289
Maximum DrawDown: -0.1473016912297861
Sharpe ratio: 3.6693520209283497


100%|██████████| 600/600 [12:17<00:00,  1.23s/it]


### Save Model

## Test Model

### Instantiate different environments

Since we have three different periods of time, we need three different environments instantiated to simulate them.

In [26]:
policy = EIIE(time_window=4, device=device,initial_features=len(features))
policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
if market.lower() == "us":
    policy.load_state_dict(torch.load("policy_EIIE_US.pt"))
elif market.lower() == "hk":
    policy.load_state_dict(torch.load("policy_EIIE_HK.pt"))
elif market.lower() == "ch":
    policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

environment_2025 = PortfolioOptimizationEnv(
    df_portfolio_2025,
    initial_amount=100000,
    comission_fee_pct=0.0025,
    time_window=4,
    features=features,
    normalize_df=None
)
# df_account_value_ppo, df_actions_ppo = DRLAgent.DRL_prediction(
#     model=model, 
#     environment = environment_2025)
DRLAgent.DRL_validation(model, environment_2025, policy=policy)
# environment_2021 = PortfolioOptimizationEnv(
#     df_portfolio_2021,
#     initial_amount=100000,
#     comission_fee_pct=0.0025,
#     time_window=50,
#     features=["close", "high", "low"],
#     normalize_df=None
# )

# environment_2022 = PortfolioOptimizationEnv(
#     df_portfolio_2022,
#     initial_amount=100000,
#     comission_fee_pct=0.0025,
#     time_window=50,
#     features=["close", "high", "low"],
#     normalize_df=None
# )

# for i, action in enumerate(environment_2025._actions_memory):
#     if not np.isnan(action).all():  # 如果 action 中不全是 NaN
#         if i < len(unique_dates):
#             current_date = unique_dates.iloc[i]
#         else:
#             current_date = '未知日期'  # 处理索引超出范围的情况
#         print(f"Action on {current_date} at step {i}: {action}")

# columns = ["date", "cash"] + TOP_BRL  
# results = []
unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)

columns = ["date", "cash"] + TOP_BRL  
results = []

shift_value = 3  # 想统一日期往后移 4 格
actions = environment_2025._actions_memory

for i, action in enumerate(actions):
    # 如果 action 全是 NaN，我们跳过
    if np.isnan(action).all():
        continue
    
    shifted_index = i + shift_value
    if shifted_index < len(unique_dates):
        current_date = unique_dates.iloc[shifted_index]
    else:
        current_date = '未知日期'
    
    # 将 action 转成百分比字符串，比如 0.123 -> "12.30%"
    action_in_percent = [f"{x*100:.2f}%" for x in action]
    
    # 构建一行 [日期, 第一列现金比例, 后面的列是各股票比例]
    row = [current_date] + action_in_percent
    results.append(row)

# 创建 DataFrame
df_action_percent = pd.DataFrame(results, columns=columns)
print("日期统一往后移 3 格后的 DataFrame：")
print(df_action_percent.tail(10))



Initial portfolio value:100000
Final portfolio value: 621692.0625
Final accumulative portfolio value: 6.216920852661133
Maximum DrawDown: -0.14665296589509424
Sharpe ratio: 3.2338697742764295


ValueError: 10 columns passed, passed data had 8 columns

In [ ]:
# unique_dates = df_portfolio['date'].drop_duplicates().sort_values().reset_index(drop=True)


### Test EIIE architecture
Now, we can test the EIIE architecture in the three different test periods. It's important no note that, in this code, we load the saved policy even though it's not necessary just to show how to save and load your model.

In [21]:
EIIE_results = {
    "training": environment._asset_memory["final"],
    "2025": {},

}

# instantiate an architecture with the same arguments used in training
# and load with load_state_dict.
policy = EIIE(time_window=50, device=device)
policy.load_state_dict(torch.load("policy_EIIE_US.pt"))

# 2020
DRLAgent.DRL_validation(model, environment_2025, policy=policy)
EIIE_results["2025"]["value"] = environment_2025._asset_memory["final"]

# # 2021
# DRLAgent.DRL_validation(model, environment_2021, policy=policy)
# EIIE_results["2021"]["value"] = environment_2021._asset_memory["final"]

# # 2022
# DRLAgent.DRL_validation(model, environment_2022, policy=policy)
# EIIE_results["2022"]["value"] = environment_2022._asset_memory["final"]

RuntimeError: Error(s) in loading state_dict for EIIE:
	size mismatch for sequential.0.weight: copying a param with shape torch.Size([2, 4, 1, 3]) from checkpoint, the shape in current model is torch.Size([2, 3, 1, 3]).
	size mismatch for sequential.2.weight: copying a param with shape torch.Size([20, 2, 1, 2]) from checkpoint, the shape in current model is torch.Size([20, 2, 1, 48]).

In [ ]:
# unique_dates = df_portfolio['date'].drop_duplicates().sort_values().reset_index(drop=True)
unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)
for i, action in enumerate(environment_2025._actions_memory):
    if not np.isnan(action).all():  # 如果 action 中不全是 NaN
        if i < len(unique_dates):
            current_date = unique_dates.iloc[i]
        else:
            current_date = '未知日期'  # 处理索引超出范围的情况
        print(f"Action on {current_date} at step {i}: {action}")

In [ ]:
filtered_df.tail()

### Test Uniform Buy and Hold
For comparison, we will also test the performance of a uniform buy and hold strategy. In this strategy, the portfolio has no remaining cash and the same percentage of money is allocated in each asset.

In [ ]:
UBAH_results = {
    "train": {},
    "2025": {},
    # "2021": {},
    # "2022": {}
}

PORTFOLIO_SIZE = len(TOP_BRL)

# train period
terminated = False
environment.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment.step(action)
UBAH_results["train"]["value"] = environment._asset_memory["final"]

# 2020
terminated = False
environment_2025.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment_2025.step(action)
UBAH_results["2025"]["value"] = environment_2025._asset_memory["final"]

# # 2021
# terminated = False
# environment_2021.reset()
# while not terminated:
#     action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
#     _, _, terminated, _ = environment_2021.step(action)
# UBAH_results["2021"]["value"] = environment_2021._asset_memory["final"]

# # 2022
# terminated = False
# environment_2022.reset()
# while not terminated:
#     action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
#     _, _, terminated, _ = environment_2022.step(action)
# UBAH_results["2022"]["value"] = environment_2022._asset_memory["final"]

### Plot graphics

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 

plt.plot(UBAH_results["train"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["training"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in training period")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2025"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2025"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2025")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2021"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2021"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2021")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2022"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2022"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2022")
plt.legend()

plt.show()

We can see that the agent is able to learn a good policy but its performance is worse the more the test period advances into the future. To get a better performance in 2022, for example, the agent should probably be trained again using more recent data.